### 국가별 법규 RAG 질의 실습

- Redis에 저장된 법규 문서 인덱스를 불러와 질문에 답하는 실습이다.
- 흐름은 기존 RAG 실습과 같이 `검색기 생성 -> 프롬프트 생성 -> LLM 생성 -> 체인 실행` 순서를 따른다.


In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Redis


In [ ]:
# 인덱스 연결

embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sbert-nli")

vectorstore = Redis.from_existing_index(
    embedding=embeddings,
    redis_url="redis://localhost:6380",
    index_name="trade_regulation_docs",
)

In [ ]:
vectorstore.similarity_search("미국 수입 통관 서류")[0].page_content

In [ ]:
def parse_query(question: str) -> dict:
    parsed = {
        "origin_country": "",
        "destination_country": "",
        "trade_direction": "",
        "commodity_name": "",
    }

    if "수출" in question:
        parsed["trade_direction"] = "export"
    elif "수입" in question:
        parsed["trade_direction"] = "import"

    if "한국" in question:
        parsed["origin_country"] = "KR"
    if "미국" in question:
        parsed["destination_country"] = "US"
    elif "일본" in question:
        parsed["destination_country"] = "JP"

    if "식품" in question:
        parsed["commodity_name"] = "food"
    elif "통관" in question:
        parsed["commodity_name"] = "customs"

    return parsed

In [ ]:
# 5단계, 검색기 생성

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
retriever.invoke("미국 수입 통관 서류")

In [ ]:
# 6단계, 프롬프트 생성

prompt_template = PromptTemplate.from_template(
    """
    귀하는 제공된 참고 문헌을 바탕으로 질문에 답하는 정보 분석 전문가입니다.
    답변 시 반드시 제시된 문맥(Context) 내의 정보만을 활용하십시오.
    만약 주어진 자료만으로 답변이 어렵다면, 추측하지 말고
    '제공된 정보로는 확인이 불가능하다'고 명확히 밝히십시오.
    모든 응답은 한국어로 작성합니다.

    #Context:
    {context}

    #Question:
    {question}

    #Answer:
    """
)

In [ ]:
# 7단계, LLM 생성

llm = ChatOpenAI(model_name="gpt-5.4-nano", temperature=0)

In [ ]:
# 8단계, 체인 생성

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

In [ ]:
question = "한국에서 미국으로 수출할 때 필요한 통관 서류는 무엇인가?"
parsed = parse_query(question)
answer = chain.invoke(question)
print(parsed)
print(answer)